# Notebook 05 - RAGAS Scoring v1 POC


In [ ]:

# ============================================================
# Notebook 05 - RAGAS Scoring Track A
# Agent Evaluation Framework v1 POC
# ============================================================

try:
    run_id
except NameError:
    run_id = "RUN-MANUAL-TEST"

try:
    environment
except NameError:
    environment = "dev"

try:
    poc_mode
except NameError:
    poc_mode = "false"
poc_mode = str(poc_mode).lower()

import csv
import datetime as dt
import io
import importlib.util
import json
import re
import types
from pathlib import Path

import yaml
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType, BooleanType

try:
    import nest_asyncio
    nest_asyncio.apply()
except Exception:
    pass

assert spark is not None, "Spark session not available."
from notebookutils import mssparkutils

LAKEHOUSE_NAME = "jacks_lakehouse"
ONELAKE_WORKSPACE_ID = "d9e51304-2b8a-4e62-b689-922e79fd76b4"
ONELAKE_LAKEHOUSE_ID = "537823c4-b83a-4a9b-9444-6039d55a4b9e"
LAKEHOUSE_DEFAULT_FILES_ROOT = "/lakehouse/default/Files"
LAKEHOUSE_FILES_ABFSS_ROOT = f"abfss://{ONELAKE_WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{ONELAKE_LAKEHOUSE_ID}/Files"
LAKEHOUSE_FILES_ROOT = LAKEHOUSE_FILES_ABFSS_ROOT
BASE_FILES_PATH = f"{LAKEHOUSE_FILES_ROOT}/agent_eval"
BASE_FILES_ABFSS_PATH = f"{LAKEHOUSE_FILES_ABFSS_ROOT}/agent_eval"
CONFIG_PATH = f"{BASE_FILES_PATH}/config"
SHARED_PATH = f"{BASE_FILES_PATH}/shared"
JUDGE_CONFIG_YAML_PATH = f"{CONFIG_PATH}/judge_config.yaml"
CALIBRATION_CSV_PATH = f"{CONFIG_PATH}/calibration_set.csv"
AUTH_UTILS_PATH = f"{SHARED_PATH}/auth_utils.py"

RESPONSES_TABLE = "agent_eval_agent_responses_staging"
SOURCE_EVIDENCE_TABLE = "agent_eval_source_evidence"
DETERMINISTIC_RESULTS_TABLE = "agent_eval_deterministic_results"
RAGAS_SCORES_TABLE = "agent_eval_ragas_scores"


def now_utc():
    return dt.datetime.now(dt.timezone.utc)


def is_onelake_path(path):
    return str(path).startswith("abfss://")


def file_exists(path):
    if is_onelake_path(path):
        return mssparkutils.fs.exists(path)
    return Path(path).is_file()


def read_text(path, max_bytes=20 * 1024 * 1024):
    if is_onelake_path(path):
        return mssparkutils.fs.head(path, max_bytes)
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def load_shared_module(path, module_name):
    if not file_exists(path):
        raise FileNotFoundError(f"Shared utility module not found: {path}")
    if is_onelake_path(path):
        module = types.ModuleType(module_name)
        module.__file__ = path
        exec(read_text(path), module.__dict__)
        return module
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


auth_utils = load_shared_module(AUTH_UTILS_PATH, "agent_eval_auth_utils")


def tokens(text):
    return {t for t in re.findall(r"[a-z0-9]+", (text or "").lower()) if len(t) > 2}


def lexical_score(response, source, expected=""):
    response_tokens = tokens(response)
    evidence_tokens = tokens(source + " " + expected)
    if not response_tokens:
        return 0.0
    return len(response_tokens & evidence_tokens) / max(1, len(response_tokens))


def load_judge_config():
    return (yaml.safe_load(read_text(JUDGE_CONFIG_YAML_PATH)) or {}).get("judge", {})


def parse_bool(value):
    if isinstance(value, bool):
        return value
    if isinstance(value, str):
        return value.strip().lower() in {"true", "1", "yes", "pass", "passed"}
    return bool(value)


def safe_float(value, default=0.0):
    if value is None:
        return default
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


def run_calibration(judge):
    if not file_exists(CALIBRATION_CSV_PATH):
        return False, "No calibration file found"
    rows = list(csv.DictReader(io.StringIO(read_text(CALIBRATION_CSV_PATH))))
    min_cases = int(judge.get("calibration_min_cases", 4))
    if len(rows) < min_cases:
        return False, f"Calibration file has {len(rows)} rows; requires at least {min_cases}"
    correct = 0
    for row in rows:
        _, _, passed, _ = llm_score(row.get("question", ""), row.get("response", ""), row.get("source_excerpt", ""), judge)
        predicted = "pass" if passed else "fail"
        if predicted == (row.get("expected_verdict") or "").lower():
            correct += 1
    accuracy = correct / len(rows)
    return accuracy >= float(judge.get("calibration_min_accuracy", 0.90)), f"calibration_accuracy={accuracy:.2f}"


def deterministic_failed_critical():
    try:
        return {
            r["test_id"]
            for r in spark.table(DETERMINISTIC_RESULTS_TABLE)
                .filter((F.col("run_id") == run_id) & (F.col("passed") == False) & (F.col("severity") == "critical"))
                .select("test_id")
                .collect()
        }
    except Exception:
        return set()


def llm_score(question, response, source, judge):
    base_url = (judge.get("base_url") or "").rstrip("/")
    allow_lexical = str(judge.get("allow_lexical_fallback", "false")).strip().lower() in {"true", "1", "yes"}
    if not base_url or "YOUR-DEVTUNNEL-URL" in base_url:
        if not allow_lexical:
            raise RuntimeError("Judge endpoint is not configured and lexical fallback is disabled")
        score = lexical_score(response, source)
        return score, score, score >= 0.70, "POC lexical scorer used because judge endpoint is not configured"
    prompt = f"""Return strict JSON with faithfulness_score, answer_relevancy_score, passed, reason.
Question: {question}
Answer: {response}
Source: {source[:6000]}"""
    try:
        payload = {
            "model": judge.get("model", "llama3.2:3b"),
            "messages": [{"role": "user", "content": prompt}],
            "temperature": float(judge.get("temperature", 0.0)),
            "response_format": {"type": "json_object"},
        }
        headers = {"Authorization": f"Bearer {judge.get('api_key', 'ollama')}", "Content-Type": "application/json"}
        api_response = auth_utils.request_with_retries("post", f"{base_url}/chat/completions", headers=headers, json=payload, timeout=120)
        api_response.raise_for_status()
        parsed = json.loads(api_response.json()["choices"][0]["message"]["content"])
        faithfulness = safe_float(parsed.get("faithfulness_score"), 0.0)
        relevancy = safe_float(parsed.get("answer_relevancy_score"), 0.0)
        passed_value = parsed.get("passed")
        if passed_value is None:
            passed_value = faithfulness >= 0.70 and relevancy >= 0.70
        return (
            faithfulness,
            relevancy,
            parse_bool(passed_value),
            parsed.get("reason", "") or "Judge returned null/blank reason",
        )
    except Exception as exc:
        if not allow_lexical:
            raise
        score = lexical_score(response, source)
        return score, score, score >= 0.70, f"Ollama judge fallback to lexical scoring after error: {str(exc)[:250]}"


schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("test_id", StringType(), False),
    StructField("agent_id", StringType(), False),
    StructField("ragas_passed", BooleanType(), False),
    StructField("faithfulness_score", DoubleType(), True),
    StructField("answer_relevancy_score", DoubleType(), True),
    StructField("source_version_hash", StringType(), True),
    StructField("judge_reason", StringType(), True),
    StructField("scored_at", TimestampType(), False),
])

judge = load_judge_config()
calibrated, calibration_reason = run_calibration(judge)
if not calibrated:
    raise RuntimeError(f"RAGAS judge calibration failed: {calibration_reason}")

skip = deterministic_failed_critical()
print(f"Judge calibration passed: {calibration_reason}")
responses = {
    r["test_id"]: r.asDict()
    for r in spark.table(RESPONSES_TABLE)
        .filter((F.col("run_id") == run_id) & (F.col("status") == "SUCCESS"))
        .filter(F.coalesce(F.col("test_origin"), F.lit("")) != "connectivity_check")
        .collect()
}
evidence = {
    r["test_id"]: r.asDict()
    for r in spark.table(SOURCE_EVIDENCE_TABLE)
        .filter(F.col("run_id") == run_id)
        .collect()
}

rows = []
for test_id, response_row in responses.items():
    ev = evidence.get(test_id, {})
    if test_id in skip:
        rows.append({
            "run_id": run_id,
            "test_id": test_id,
            "agent_id": response_row["agent_id"],
            "ragas_passed": False,
            "faithfulness_score": 0.0,
            "answer_relevancy_score": 0.0,
            "source_version_hash": ev.get("version_hash"),
            "judge_reason": "Skipped by Notebook 05 because a critical deterministic rule failed",
            "scored_at": now_utc(),
        })
        continue
    faithfulness, relevancy, passed, reason = llm_score(response_row.get("question", ""), response_row.get("agent_response", ""), ev.get("raw_excerpt", ""), judge)
    rows.append({
        "run_id": run_id,
        "test_id": test_id,
        "agent_id": response_row["agent_id"],
        "ragas_passed": bool(passed),
        "faithfulness_score": faithfulness,
        "answer_relevancy_score": relevancy,
        "source_version_hash": ev.get("version_hash"),
        "judge_reason": reason,
        "scored_at": now_utc(),
    })

if not rows:
    raise RuntimeError("RAGAS scoring produced no rows")
spark.createDataFrame([Row(**r) for r in rows], schema=schema).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(RAGAS_SCORES_TABLE)
print(f"RAGAS scoring complete. rows={len(rows)} skipped_by_deterministic={len(skip)}")

try:
    from notebookutils import mssparkutils
    mssparkutils.notebook.exit("PASS")
except ImportError:
    pass
